### **Fih Segmentation Pipeline**

#### Import Packages

In [1]:
import cv2
import numpy as np
from dataloader import MOVE
import os
import random


import matplotlib.pyplot as plt
from matplotlib.pyplot import *
import scipy
from scipy.stats import norm
from scipy.stats import *
from scipy.optimize import curve_fit

import math

#### Global Variables

In [2]:
RANDOM_SAMPLE = False

#### Helper Functions

In [3]:
def generate_path(rando : bool) -> str:
    if rando==True:
        ROOT = os.path.dirname(os.getcwd())
        DATA_DIR = os.path.join(ROOT, 'data')
        FIN_DIR = os.path.join(DATA_DIR, random.choice(list(filter(lambda x: os.path.isdir(os.path.join(DATA_DIR, x)), os.listdir(DATA_DIR)))))
        vp = os.path.join(FIN_DIR, random.choice(list(filter(lambda x: os.path.splitext(x)[1]=='.mov', os.listdir(FIN_DIR)))))
    else:
        # 1/32" tester
        #vp = r'C:\Users\aatid\Downloads\Interesting Air Cavity Data\Interesting Air Cavity Data\TH32\TH32ST17A3D1000TS0_2.mov'

        # 1/16" tester
        #vp = r'C:\Users\aatid\Downloads\Interesting Air Cavity Data\Interesting Air Cavity Data\TH16\TH16ST25A3D500TS0_2.mov'

        # 3/32" tester (boring)
        #vp = r'C:\Users\aatid\Downloads\Interesting Air Cavity Data\Interesting Air Cavity Data\TH332\TH332ST25A3D750TS0_2.mov'

        # 3/32" tester (bubbly)
        vp = r'C:\Users\aatid\Downloads\Interesting Air Cavity Data\Interesting Air Cavity Data\TH332\TH332ST17A3D1000TS0_2.mov'

        # 1/8" tester
        #vp = r'C:\Users\aatid\Downloads\Interesting Air Cavity Data\Interesting Air Cavity Data\TH8\TH8ST17A3D1000TS0_2.mov'
    print(vp)

    return vp

In [4]:
# some video helper functions - play two videos stacked vertically, play a series of frames that's not a MOVE object, or
# create a linear bounding box based on contours and overlay it onto an image

def compare_videos(frames1, frames2, crop = None, doubletime = False, slow = False, ultraslow = False):
    divider = 255 * np.ones((10, frames1[0].shape[1]))
    frames = []

    if len(frames1) != len(frames2):
        frames1 = frames1[:min([len(frames1), len(frames2)])]
        frames2 = frames2[:min([len(frames1), len(frames2)])]

    for i in range(len(frames1)):
        frame1 = frames1[i][:crop, :]
        frame2 = frames2[i][:crop, :]
        joined = np.concat((frame1, divider, frame2), axis = 0)
        frames.append(joined)
    
    for frame in frames:
            cv2.imshow("Video Frame", frame)
            if doubletime:
                if cv2.waitKey(12) & 0xFF == ord('q'):
                    break
            elif slow:
                if cv2.waitKey(40) & 0xFF == ord('q'):
                    break
            elif ultraslow:
                if cv2.waitKey(60) & 0xFF == ord('q'):
                    break
            else:
                if cv2.waitKey(25) & 0xFF == ord('q'):
                    break
                 
        
    cv2.destroyAllWindows()

def play_video(frames, doubletime=False, slow=False, ultraslow=False):
    for frame in frames:
        cv2.imshow("Video Frame", frame)
        if doubletime:
            if cv2.waitKey(12) & 0xFF == ord('q'):
                break
        elif slow:
            if cv2.waitKey(40) & 0xFF == ord('q'):
                break
        elif ultraslow:
            if cv2.waitKey(60) & 0xFF == ord('q'):
                break
        else:
            if cv2.waitKey(25) & 0xFF == ord('q'):
                break
                 
    cv2.destroyAllWindows()

In [ ]:
# some bounding helper functions - create a linear (or other) fit based on contours and overlay it onto an existing video

# this is relatively efficient at the cost of expecting a specific structure for the linear fit you want to overlay:
# exactly 2 pixels per row, only slightly changing their position frame to frame, not close to the edges
def linear_bounding_overlay(frames1, frames2, color = (0, 0, 255), nsearch = 10):
    output_frames = []
    left = nsearch + 1
    right = nsearch + 1
    row = 0
    still_fin = True

    for frame in range(len(frames1)):
        temp_frame = cv2.cvtColor(frames1[frame], cv2.COLOR_BGR2RGB)
        row = 0
        still_fin = True
        while row < temp_frame.shape[0] and still_fin:
            px_found = 0
            if np.isin(255, frames2[frame][row]):
            #fingers crossed efficiency
                for col in range(left + nsearch, left - nsearch, -1):
                    if frames2[frame][row][col] == 255:
                        temp_frame[row][col] = color
                        left = col
                        px_found += 1
                for col in range(right - nsearch, right + nsearch):
                    if frames2[frame][row][col] == 255:
                        temp_frame[row][col] = color
                        right = col
                        px_found += 1
                if px_found < 2:
                    count = 0
                    for col in range(temp_frame.shape[1]):
                        if frames2[frame][row][col] == 255:
                            temp_frame[row][col] = color
                            if count == 0: left = col
                            if count == 1: right = col
            else:
                still_fin = False
            row += 1
        output_frames.append(temp_frame)
    
    return output_frames

### **Cavity Segmentation Pipeline**

#### Data Loading & Binary Masking

In [5]:
# load video and convert to grayscale

vid_path = generate_path(False)
recording = MOVE(vid_path)
recording.play_video( )
recording.to_gray(inplace=True)
#recording.change_contrast(alpha=1, beta=-40, inplace=True)
play_video(recording.frames, ultraslow=True)

C:\Users\aatid\Downloads\Interesting Air Cavity Data\Interesting Air Cavity Data\TH332\TH332ST17A3D1000TS0_2.mov


In [7]:
# remove background and convert to binary image

recording.change_contrast(alpha=2, beta=-10, inplace=True)
recording.remove_avg(skipframes=100)
#recording.play_video()

In [8]:
# create a copy of the video

out:MOVE = recording.ranger(minval=50, maxval=240)
out.median_blur(inplace=True)
#out.play_video()

In [9]:
# fill contained holes in binary mask

filled = out.hole_filler()
#filled.play_video()

In [9]:
cropped = out.copy()
cropped = cropped.crop(row_start = 450, row_end = 950)
MOVE.show(cropped.frames[2])

#### Waterline Detection

In [10]:
# find background & horizontally blur

recording = MOVE(vid_path)
recording.to_gray(inplace=True)
background = recording.get_avg()
background = cv2.GaussianBlur(background, (1111, 1), sigmaX=32000, borderType=cv2.BORDER_REFLECT)

In [11]:
# use kmeans clustering to segment the background into 3 regions of different brightness

Z = np.float32(background.reshape((-1, 3)))

criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
_, labels, centers = cv2.kmeans(Z, 3, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)

centers = np.uint8(centers)
quantized = centers[labels.flatten()]
quantized = quantized.reshape(background.shape)
MOVE.show(quantized)

In [12]:
# determine pixel values corresponding to each color

convenience = [centers[0][0], centers[1][0], centers[2][0]]

light_color = np.uint8(max(convenience))
convenience.remove(light_color)
mid_color = np.uint8(max(convenience))
convenience.remove(mid_color)
dark_color = convenience[0]

In [13]:
# use a single column of pixels taken from the center of the background to determine the waterline

slice = quantized[:, quantized.shape[1]//2]
flatslice = slice.flatten()

waterline = 0
count = 0
i = 0
while count < 2 and i < flatslice.size:
    i += 1
    if flatslice[i] == mid_color and flatslice[i + 1] == light_color:
        count += 1
        waterline = i
        #print(waterline)

In [37]:
# new thing

slice = quantized[:, quantized.shape[1]//2]
flatslice = slice.flatten()

waterline = 0
i = 0
while i < flatslice.size:
    i += 1
    if flatslice[i] == mid_color and flatslice[i + 1] == light_color:
        waterline = i
        print(waterline)
        break

552


In [14]:
# new new thing

slice = quantized[:, quantized.shape[1]//2]
flatslice = slice.flatten()

waterline = 0
count = 0
i = 0
while count < 2 and i < flatslice.size:
    i += 1
    if flatslice[i] == dark_color and flatslice[i + 1] == mid_color:
        count += 1
        waterline = i
        print(waterline)

172
545


In [15]:
# crop the video at the waterline

cropped = filled.copy()
cropped = cropped.crop(row_start = waterline)
#cropped.play_video()

#### Image Segmentation - Fin Removal

In [16]:
# remove the fin and play video

fin_removed = cropped.copy()
fin_removed = fin_removed.fin_removal(ksize = (6, 18))
#fin_removed.play_video()

In [38]:
# video comparison - masked, cropped video and mask after fin removal
compare_videos(cropped.frames, fin_removed.frames, crop=450, doubletime=True)

In [17]:
# create air cavity contours and play video

cavity_contoured = fin_removed.contouring(thresholding = True, threshold = 270)
#cavity_contoured.play_video()

In [41]:
# video comparison - masked, cropped video and air cavity contours
compare_videos(cropped.frames, cavity_contoured.frames, crop=400)

#### Image Segmentation - Fin Retrieval

In [37]:
# retrieve the fin and play video

fin_retrieved = cropped.copy()
fin_retrieved = fin_retrieved.fin_retrieval(horiz_ksize = (3, 12), vert_ksize = (8, 4))
#fin_retrieved.play_video()

In [17]:
# video comparison - masked, cropped video and mask after fin retrieval
compare_videos(cropped.frames, fin_retrieved.frames)

In [18]:
# create fin contours and play video

fin_contoured = fin_retrieved.contouring(thresholding = False)
#fin_contoured.play_video()

In [16]:
# video comparison - masked, cropped video and fin contours
compare_videos(cropped.frames, fin_contoured.frames, crop=400)

### **Air Cavity Parameter Determination**

In [50]:
# determine leftmost and rightmost pixels in a frame

cavity_contoured = cavity_contoured
frame = cavity_contoured[500]

leftmost = 0
for i in range(frame.shape[1]):
    if np.isin([255], cavity_contoured.frames[i])[0]:
        leftmost = i
        print(leftmost)
        break

rightmost = frame.shape[1]
for i in range(frame.shape[1]):
    j = frame.shape[1] - i
    if np.isin([255], cavity_contoured.frames[j])[0]:
        rightmost = j
        print(rightmost)
        break

87


IndexError: list index out of range

In [ ]:
print(np.isin([0], cavity_contoured.frames[100]))

[ True]


TRYING TO EXTRACT PLOTS FROM FRAMES

**FIN TRACKING OF MISERY AND WOE**

In [19]:
#def myfit(x, offset, a, b, c, d, e):
#    x_corr = x - offset
#    return a*np.power(x_corr, 4) + b*np.power(x_corr, 3) + c*np.power(x_corr, 2) + d*np.power(x_corr, 1) + e

def myfit(x, a, b):
    return a*x + b

def frame_to_xy(frame):
    y = []
    x = []
    half_width = []

    for row in range(0, frame.shape[0]):
        active_row = frame[row]
        if np.isin(255, active_row):
            leftmost = -1
            rightmost = -1
            col = 0
            while leftmost < 0 and col < frame.shape[1]:
                if active_row[col] == 255:
                    leftmost = col
                col += 1
            col = frame.shape[1] - 1
            while rightmost < 0 and col >= 0:
                if active_row[col] == 255:
                    rightmost = col
                col -= 1
            x.append(float(row))
            center =  (leftmost + rightmost) / 2.0
            y.append(float(center))
            half_width.append((rightmost - leftmost) / 2.0)
    
    return x, y, half_width

def produce_frame(shape, rows, centers, half_widths, offset=2):
    output_frame = np.zeros(shape)
    if len(half_widths) > 0: avg_half_width = int(sum(half_widths) / len(half_widths))
    else: avg_half_width = 0

    for n in rows:
        n = int(n)
        center = centers[n]
        half_width = avg_half_width
        #try:
        #    if half_widths[n] > 0: half_width = half_widths[n]
        #    else: half_width = avg_half_width
        #except:
        #    center = int(centers[n]) 
        #    half_width = avg_half_width
        p = int(center + half_width + offset)
        m = int(center - half_width - offset)
        if (p) < shape[1] and (p) >= 0: 
            output_frame[n][p] = 255
        if (m) < shape[1] and (m) >= 0: 
            output_frame[n][m] = 255
    
    return output_frame

def param_update(saved_params, new_row):
    max = saved_params.shape[0] - 1
    for i in range(0, max):
        saved_params[i] = saved_params[i + 1]
    saved_params[max] = new_row

    return saved_params

In [20]:
test_vid = fin_contoured.copy()

lowest_point = -1
output_frames = []
p0 = []
saved_params = np.zeros((5, 2))

for i in range(0, len(test_vid.frames)):
    active_frame = test_vid.frames[i]

    x, y, half_widths = frame_to_xy(active_frame)

    if i == 0:
        half_widths = np.rint(half_widths)
        x = np.array(x)
        yerr = np.array(y) / 10
        if len(x) > 0: 
            zero = np.zeros(int(min(x)))
            half_widths = np.concatenate((zero, half_widths))
        lowest_point = max(x)

        (popt, pcov) = curve_fit(myfit, x, y, sigma=yerr, absolute_sigma=True)
        saved_params = param_update(saved_params, popt)
        p0 = popt

    elif i < 5:
        half_widths = np.rint(half_widths)
        #if len(x) > 0: temp_lowest = min(lowest_point, max(x))
        #else: temp_lowest = lowest_point
        x = np.array(x)
        yerr = 0.1 * np.array(y)
        if len(x) > 0: 
            zero = np.zeros(int(min(x)))
            half_widths = np.concatenate((zero, half_widths))

        try:
            (popt, pcov) = curve_fit(myfit, x, y, sigma=yerr, p0=p0, absolute_sigma=True)
            #(popt, pcov) = curve_fit(myfit, x, y, sigma=yerr, absolute_sigma=True)
            saved_params = param_update(saved_params, popt)
            p0 = popt
        except:
            popt = p0
            print(i)
        

    else:
        half_widths = np.rint(half_widths)
        #if len(x) > 0: temp_lowest = min(lowest_point, max(x))
        #else: temp_lowest = lowest_point
        x = np.array(x)
        yerr = 0.1 * np.array(y)
        if len(x) > 0: 
            zero = np.zeros(int(min(x)))
            half_widths = np.concatenate((zero, half_widths))

        try:
            seed = np.mean(saved_params, axis=0)
            (popt, pcov) = curve_fit(myfit, x, y, sigma=yerr, p0=seed, absolute_sigma=True)
            #(popt, pcov) = curve_fit(myfit, x, y, sigma=yerr, absolute_sigma=True)
            saved_params = param_update(saved_params, popt)
            p0 = popt
        except:
            popt = p0
            print(i)

    rows = np.arange(lowest_point)
    centers = myfit(rows, *popt)
    centers = np.rint(centers)
        
    frame = produce_frame(shape=active_frame.shape, rows=rows, centers=centers, half_widths=np.zeros_like(half_widths), offset=12)
    output_frames.append(frame)

166
168
169


In [55]:
compare_videos(output_frames, cropped.frames, crop=450)

In [57]:
test = cropped.copy()
test1 = test.frames[500]
test1 = cv2.cvtColor(test1, cv2.COLOR_BGR2RGB)
test1[100:110, 100:110] = (255, 0, 0)
MOVE.show(test1)

In [ ]:
overlay_vid = linear_bounding_overlay(cropped.frames, output_frames)

In [ ]:
yikes = linear_bounding_overlay(cropped.frames, output_frames, nsearch=40)

In [31]:
play_video(yikes)

**Connected components experiment**

In [42]:
test_vid = fin_removed.copy()
test_frame = test_vid.frames[550]
MOVE.show(test_frame)

## **Scrapyard**

In [ ]:
# circle detection

contour_dupe = fin_removed.copy()
test_frame = contour_dupe.frames[550]
test_frame = test_frame.astype(np.uint8)
MOVE.show(test_frame)

blank = np.zeros_like(test_frame)
blank = cv2.cvtColor(blank, cv2.COLOR_BGR2RGB)

circles = cv2.HoughCircles(test_frame, cv2.HOUGH_GRADIENT, dp=1, minDist=5, param1=40, param2=20, minRadius=10, maxRadius=60)

if circles is not None:
    circles = np.uint16(np.around(circles))

    x, y, r = circles[0][0]
    print(r)
    test_frame = cv2.cvtColor(test_frame, cv2.COLOR_BGR2RGB)
    cv2.circle(test_frame, (x, y), r, (0, 200, 0), -2)  # Circle outline
else:
    cv2.circle(test_frame, (100, 100), 50, (0, 255, 0), 2)
    print("no circles :(")

MOVE.show(test_frame)